<a href="https://colab.research.google.com/github/iabbey-commits/Financial-Analysis-Agent/blob/main/Financial_Analysis_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Architecture for the Financial Analysis Agent


In [ ]:
from graphviz import Digraph

def financial_agent_architecture():
    agent5 = Digraph(format='png')
    agent5.attr(rankdir='TD', size='20,15')

    # Global node style
    agent5.attr('node', shape='rectangle', style='filled', color='lightblue', fontname='Helvetica')

    # Nodes
    agent5.node('A', 'KNOWLEDGE PREPARATION LAYER', shape='rectangle', style='filled', color='blue')

    agent5.node('A1', 'Data Ingestion')

    agent5.node('A2', 'Table Segmentation')

    agent5.node('A3', 'KPI Extraction')

    agent5.node('A4', 'Vector Indexing')

    agent5.node('B', 'AGENT LAYER', shape='circle', style='filled', fillcolor='magenta')

    agent5.node('C', 'User Query')

    agent5.node('D', 'Query Router')

    agent5.node('E', 'Analysis Detector', shape='circle', style='filled', fillcolor='lightgreen')

    agent5.node('E1', 'Within Section')

    agent5.node('E2', 'Cross Section')

    agent5.node('E3', 'Monthly')

    agent5.node('F', 'Multi-Dimensional RAG')

    agent5.node('G', 'Verified Financial Records')

    agent5.node('H', 'Python Financial Analysis', shape='circle', style='filled', fillcolor='yellow')

    agent5.node('H1', 'Retrieval')

    agent5.node('H2', 'Variance')

    agent5.node('H3', 'Comparison')

    agent5.node('H4', 'Diagnostic')

    agent5.node('I', 'Pandas DataFrame')

    agent5.node('J', 'Verified Context')

    agent5.node('K', 'NLG Agent', shape='circle', style='filled', fillcolor='orange')

    agent5.node('L', 'Final Answer')

    # Edges
    agent5.edge('A', 'A1')
    agent5.edge('A', 'A2')
    agent5.edge('A', 'A3')
    #agent5.edge('A1', 'A2')
    #agent5.edge('A2', 'A3')
    agent5.edge('A1', 'A4')
    agent5.edge('A2', 'A4')
    agent5.edge('A3', 'A4')
    agent5.edge('A4', 'B')
    agent5.edge('B', 'C')
    agent5.edge('C', 'D')
    agent5.edge('D', 'E')
    agent5.edge('E', 'E1')
    agent5.edge('E', 'E2')
    agent5.edge('E', 'E3')
    agent5.edge('E1', 'F')
    agent5.edge('E2', 'F')
    agent5.edge('E3', 'F')
    agent5.edge('F', 'G')
    agent5.edge('G', 'H')
    agent5.edge('H', 'H1')
    agent5.edge('H', 'H2')
    agent5.edge('H', 'H3')
    agent5.edge('H', 'H4')
    agent5.edge('H1', 'I')
    agent5.edge('H2', 'I')
    agent5.edge('H3', 'I')
    agent5.edge('H4', 'I')
    agent5.edge('I', 'J')
    agent5.edge('J', 'K')
    agent5.edge('K', 'L')

    return agent5


# Generate and save diagram
diagram = financial_agent_architecture()
diagram.render('Finance_agent_architecture', view=True)


In [ ]:
# View Diagram
from IPython.display import Image
Image('Finance_agent_architecture.png')

Implemented Code for the Financial Analysis Agent.


In [ ]:
!pip install pandas numpy faiss-cpu sentence-transformers groq openpyxl python-dotenv

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import os
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

In [ ]:
from groq import Groq

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [ ]:
import pandas as pd

file_name = list(uploaded.keys())[0]

df = pd.read_excel(file_name, header=None)

df.head(20)

In [ ]:
# ============================================================
# MULTI-SHEET FINANCIAL RAG DATASET
# ============================================================

import numpy as np


# ============================================================
# 1. WORKBOOK SETTINGS
# ============================================================

file_name = list(uploaded.keys())[0]

excel_file = pd.ExcelFile(file_name)

print("Available sheets:")
print(excel_file.sheet_names)


# ============================================================
# 2. SHEETS TO INCLUDE
# ============================================================
#
# We include:
#   Full-Year
#   1 through 12
#
# "Monthly" is excluded by default because it may duplicate
# information represented by the 12 individual month sheets.
#
# You can add "Monthly" later if it contains unique information.
# ============================================================

month_map = {
    "1": "January",
    "2": "February",
    "3": "March",
    "4": "April",
    "5": "May",
    "6": "June",
    "7": "July",
    "8": "August",
    "9": "September",
    "10": "October",
    "11": "November",
    "12": "December"
}

wanted_sheets = ["Full-Year"] + list(month_map.keys())

available_sheets = excel_file.sheet_names

included_sheets = [
    sheet
    for sheet in wanted_sheets
    if sheet in available_sheets
]

missing_sheets = [
    sheet
    for sheet in wanted_sheets
    if sheet not in available_sheets
]

print("\nIncluded sheets:")
print(included_sheets)

if missing_sheets:
    print("\nMissing expected sheets:")
    print(missing_sheets)


# ============================================================
# 3. COLUMN MAPPING
# ============================================================

KPI_COL = 3       # D
BUDGET_COL = 4    # E
PRIOR_COL = 7     # H
ACTUAL_COL = 10   # K


# ============================================================
# 4. TEXT NORMALIZATION
# ============================================================

def normalize_text(value):

    if pd.isna(value):
        return ""

    text = str(value)

    text = text.replace("\xa0", " ")

    text = " ".join(text.split())

    return text.strip()


# ============================================================
# 5. NUMBER CLEANING
# ============================================================

def clean_number(value):

    if pd.isna(value):
        return None

    if isinstance(
        value,
        (int, float, np.integer, np.floating)
    ):

        if pd.isna(value):
            return None

        return float(value)

    value = str(value).strip()

    if value == "":
        return None

    value = (
        value
        .replace(",", "")
        .replace("$", "")
    )

    if value.endswith("%"):
        value = value[:-1].strip()

    try:
        return float(value)

    except (ValueError, TypeError):
        return None


# ============================================================
# 6. EXTRACT TABLES FROM ONE SHEET
# ============================================================

def extract_tables_with_metadata(df):

    tables = []

    current = []
    start_row = None

    for row_index, row in df.iterrows():

        # Blank row ends table
        if row.isnull().all():

            if current:

                table_df = pd.DataFrame(current)

                tables.append({
                    "start_row": start_row + 1,
                    "end_row": row_index,
                    "data": table_df
                })

                current = []
                start_row = None

        else:

            if start_row is None:
                start_row = row_index

            current.append(row.tolist())

    # Final table
    if current:

        table_df = pd.DataFrame(current)

        tables.append({
            "start_row": start_row + 1,
            "end_row": len(df),
            "data": table_df
        })

    return tables


# ============================================================
# 7. IDENTIFY TABLE SECTION
# ============================================================

def get_table_section(table_df):

    if table_df.empty:
        return "Unclassified"

    # First row, column D
    if len(table_df.columns) > KPI_COL:

        first_value = table_df.iloc[
            0,
            KPI_COL
        ]

        section = normalize_text(
            first_value
        )

        if section:
            return section

    # Fallback: first few rows
    for i in range(
        min(5, len(table_df))
    ):

        value = table_df.iloc[
            i,
            KPI_COL
        ]

        section = normalize_text(value)

        if section:
            return section

    return "Unclassified"


# ============================================================
# 8. DETERMINE PERIOD NAME
# ============================================================

def get_period(sheet_name):

    if sheet_name == "Full-Year":
        return "Full-Year"

    return month_map.get(
        str(sheet_name),
        str(sheet_name)
    )


# ============================================================
# 9. DETERMINE WHETHER ROW CONTAINS FINANCIAL DATA
# ============================================================

def is_financial_row(row):

    if len(row) <= ACTUAL_COL:
        return False

    name = normalize_text(
        row.iloc[KPI_COL]
    )

    if not name:
        return False

    budget = clean_number(
        row.iloc[BUDGET_COL]
    )

    prior = clean_number(
        row.iloc[PRIOR_COL]
    )

    actual = clean_number(
        row.iloc[ACTUAL_COL]
    )

    if (
        budget is None
        and prior is None
        and actual is None
    ):
        return False

    return True


# ============================================================
# 10. FINANCIAL ANALYSIS
# ============================================================

def calculate_variance(
    actual,
    comparison
):

    if (
        actual is None
        or comparison is None
    ):
        return None

    return actual - comparison


def calculate_variance_pct(
    actual,
    comparison
):

    if (
        actual is None
        or comparison is None
        or comparison == 0
    ):
        return None

    return (
        (actual - comparison)
        / abs(comparison)
    ) * 100


# ============================================================
# 11. EXTRACT ONE SHEET
# ============================================================

def extract_sheet_records(
    sheet_name
):

    df_sheet = pd.read_excel(
        file_name,
        sheet_name=sheet_name,
        header=None
    )

    tables = extract_tables_with_metadata(
        df_sheet
    )

    period = get_period(sheet_name)

    sheet_records = []

    for table_number, table_info in enumerate(
        tables,
        start=1
    ):

        table_df = table_info["data"]

        section = get_table_section(
            table_df
        )

        start_row = table_info[
            "start_row"
        ]

        for local_index, row in table_df.iterrows():

            if not is_financial_row(row):
                continue

            name = normalize_text(
                row.iloc[KPI_COL]
            )

            budget = clean_number(
                row.iloc[BUDGET_COL]
            )

            prior = clean_number(
                row.iloc[PRIOR_COL]
            )

            actual = clean_number(
                row.iloc[ACTUAL_COL]
            )

            source_row = (
                start_row
                + local_index
            )

            variance_budget = (
                calculate_variance(
                    actual,
                    budget
                )
            )

            variance_budget_pct = (
                calculate_variance_pct(
                    actual,
                    budget
                )
            )

            variance_prior = (
                calculate_variance(
                    actual,
                    prior
                )
            )

            variance_prior_pct = (
                calculate_variance_pct(
                    actual,
                    prior
                )
            )

            # ------------------------------------------------
            # Unique record identity
            # ------------------------------------------------
            #
            # Period + Section + KPI
            #
            # This means:
            #
            # January | Operating Revenue | F&B Division
            #
            # is different from:
            #
            # February | Operating Revenue | F&B Division
            # ------------------------------------------------

            record_id = (
                f"{period} | "
                f"{section} | "
                f"{name}"
            )

            record = {

                "record_id": record_id,

                "period": period,

                "sheet": sheet_name,

                "section": section,

                "name": name,

                "actual": actual,

                "budget": budget,

                "prior": prior,

                "variance_to_budget":
                    variance_budget,

                "variance_to_budget_pct":
                    variance_budget_pct,

                "variance_to_prior":
                    variance_prior,

                "variance_to_prior_pct":
                    variance_prior_pct,

                "source_table":
                    table_number,

                "source_row":
                    source_row
            }

            sheet_records.append(
                record
            )

    return sheet_records


# ============================================================
# 12. EXTRACT ALL INCLUDED SHEETS
# ============================================================

all_financial_records = []

sheet_counts = {}

for sheet_name in included_sheets:

    records = extract_sheet_records(
        sheet_name
    )

    all_financial_records.extend(
        records
    )

    sheet_counts[sheet_name] = len(
        records
    )


# ============================================================
# 13. REMOVE ONLY EXACT DUPLICATES
# ============================================================

def remove_exact_duplicates(
    records
):

    seen = set()
    unique_records = []

    for record in records:

        key = (
            record["period"],
            normalize_text(
                record["section"]
            ).lower(),
            normalize_text(
                record["name"]
            ).lower(),
            record["actual"],
            record["budget"],
            record["prior"]
        )

        if key not in seen:

            seen.add(key)

            unique_records.append(
                record
            )

    return unique_records


financial_records = (
    remove_exact_duplicates(
        all_financial_records
    )
)


# ============================================================
# 14. REPORT
# ============================================================

print("\n" + "=" * 70)
print("MULTI-SHEET FINANCIAL EXTRACTION")
print("=" * 70)

print(
    "Total raw records:",
    len(all_financial_records)
)

print(
    "Total unique records:",
    len(financial_records)
)

print("\nRecords by sheet:")

for sheet_name, count in sheet_counts.items():

    print(
        f"{sheet_name}: {count}"
    )

In [ ]:
print("\n" + "=" * 70)
print("PERIODS CAPTURED")
print("=" * 70)

periods = sorted(
    set(
        r["period"]
        for r in financial_records
    ),
    key=lambda x: (
        0 if x == "Full-Year"
        else list(month_map.values()).index(x) + 1
    )
)

for period in periods:
    print(period)

In [ ]:
print("\n" + "=" * 70)
print("PERIODS CAPTURED")
print("=" * 70)

periods = sorted(
    set(
        r["period"]
        for r in financial_records
    ),
    key=lambda x: (
        0 if x == "Full-Year"
        else list(month_map.values()).index(x) + 1
    )
)

for period in periods:
    print(period)

In [ ]:
# ============================================================
# MULTI-PERIOD RAG EMBEDDING INDEX
# ============================================================

from sentence_transformers import SentenceTransformer
# import numpy as np


embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)


def build_rag_document(record):

    return " | ".join([
        f"Period: {record['period']}",
        f"Sheet: {record['sheet']}",
        f"Section: {record['section']}",
        f"KPI: {record['name']}"
    ])


rag_documents = [
    build_rag_document(record)
    for record in financial_records
]


rag_embeddings = embedding_model.encode(
    rag_documents,
    convert_to_numpy=True,
    normalize_embeddings=True
)


print(
    "RAG index contains",
    len(rag_documents),
    "financial records."
)

In [ ]:
# ============================================================
# MONTH DETECTION
# ============================================================

MONTH_ALIASES = {
    "jan": "January",
    "january": "January",

    "feb": "February",
    "february": "February",

    "mar": "March",
    "march": "March",

    "apr": "April",
    "april": "April",

    "may": "May",

    "jun": "June",
    "june": "June",

    "jul": "July",
    "july": "July",

    "aug": "August",
    "august": "August",

    "sep": "September",
    "sept": "September",
    "september": "September",

    "oct": "October",
    "october": "October",

    "nov": "November",
    "november": "November",

    "dec": "December",
    "december": "December"
}


def detect_requested_period(query):

    query_clean = normalize_text(
        query
    ).lower()

    # Full year
    if (
        "full year" in query_clean
        or "annual" in query_clean
        or "year end" in query_clean
    ):
        return "Full-Year"

    # Individual month
    for alias, month in MONTH_ALIASES.items():

        if re.search(
            rf"\b{re.escape(alias)}\b",
            query_clean
        ):
            return month

    return None

In [ ]:
# ============================================================
# MONTHLY RAG RETRIEVAL - COMPLETE VERSION
# ============================================================

import re
# import numpy as np


# ============================================================
# 1. MONTH DEFINITIONS
# ============================================================

MONTH_MAP = {
    "1": "January",
    "2": "February",
    "3": "March",
    "4": "April",
    "5": "May",
    "6": "June",
    "7": "July",
    "8": "August",
    "9": "September",
    "10": "October",
    "11": "November",
    "12": "December"
}


MONTH_ORDER = {
    month: i
    for i, month in enumerate(
        MONTH_MAP.values(),
        start=1
    )
}


# ============================================================
# 2. NORMALIZE TEXT
# ============================================================

def rag_normalize(text):

    if text is None:
        return ""

    text = str(text)

    text = text.replace("\xa0", " ")

    text = " ".join(text.split())

    text = text.lower().strip()

    # Standardize common terms
    text = text.replace(
        "f&b",
        "food and beverage"
    )

    text = text.replace(
        "f & b",
        "food and beverage"
    )

    text = text.replace(
        "food & beverage",
        "food and beverage"
    )

    text = text.replace(
        "adr",
        "average rate"
    )

    return text


# ============================================================
# 3. DETECT SECTION FROM USER QUESTION
# ============================================================

def detect_requested_section(
    query,
    records
):

    query_clean = rag_normalize(query)

    # --------------------------------------------------------
    # Get actual sections present in the workbook
    # --------------------------------------------------------

    available_sections = sorted(
        set(
            rag_normalize(
                r.get("section", "")
            )
            for r in records
            if r.get("section")
        ),
        key=len,
        reverse=True
    )

    # --------------------------------------------------------
    # 1. Exact section mentioned
    # --------------------------------------------------------

    for section in available_sections:

        if (
            section
            and section in query_clean
        ):
            return section

    # --------------------------------------------------------
    # 2. Revenue
    # --------------------------------------------------------

    if "total revenue" in query_clean:

        for section in available_sections:

            if section == "total revenue":

                return section

    if "revenue" in query_clean:

        for section in available_sections:

            if section == "operating revenue":

                return section

    # --------------------------------------------------------
    # 3. Expenses
    # --------------------------------------------------------

    if (
        "expense" in query_clean
        or "expenses" in query_clean
    ):

        for section in available_sections:

            if "departmental expenses" in section:

                return section

    # --------------------------------------------------------
    # 4. Profit
    # --------------------------------------------------------

    if "profit" in query_clean:

        for section in available_sections:

            if "departmental profit" in section:

                return section

    # --------------------------------------------------------
    # 5. Salary / wages
    # --------------------------------------------------------

    if (
        "salary" in query_clean
        or "salaries" in query_clean
        or "wage" in query_clean
        or "wages" in query_clean
    ):

        for section in available_sections:

            if (
                "salary" in section
                or "wage" in section
            ):

                return section

    # --------------------------------------------------------
    # 6. Statistics
    # --------------------------------------------------------

    if (
        "occupancy" in query_clean
        or "average rate" in query_clean
        or "revpar" in query_clean
    ):

        for section in available_sections:

            if section == "statistics":

                return section

    return None


# ============================================================
# 4. DETECT KPI FROM USER QUESTION
# ============================================================

def detect_kpi_candidates(
    query,
    records
):

    query_clean = rag_normalize(query)

    candidates = []

    for index, record in enumerate(records):

        name = rag_normalize(
            record.get("name", "")
        )

        if not name:
            continue

        # ----------------------------------------------------
        # Direct KPI match
        # ----------------------------------------------------

        if name in query_clean:

            candidates.append(index)

    return candidates


# ============================================================
# 5. DETECT REQUESTED MONTH
# ============================================================

def detect_requested_month(query):

    query_clean = rag_normalize(query)

    month_aliases = {

        "jan": "January",
        "january": "January",

        "feb": "February",
        "february": "February",

        "mar": "March",
        "march": "March",

        "apr": "April",
        "april": "April",

        "may": "May",

        "jun": "June",
        "june": "June",

        "jul": "July",
        "july": "July",

        "aug": "August",
        "august": "August",

        "sep": "September",
        "sept": "September",
        "september": "September",

        "oct": "October",
        "october": "October",

        "nov": "November",
        "november": "November",

        "dec": "December",
        "december": "December"
    }

    for alias, month in month_aliases.items():

        if re.search(
            rf"\b{re.escape(alias)}\b",
            query_clean
        ):
            return month

    return None


# ============================================================
# 6. DETERMINE WHETHER QUERY REQUESTS MONTHLY ANALYSIS
# ============================================================

def is_monthly_analysis_query(query):

    query_clean = rag_normalize(query)

    analysis_terms = [
        "each month",
        "every month",
        "monthly",
        "by month",
        "month by month",
        "across the months",
        "throughout the year",
        "monthly performance",
        "monthly trend",
        "trend across",
        "12 months",
        "all months"
    ]

    for term in analysis_terms:

        if term in query_clean:
            return True

    return False


# ============================================================
# 7. RETRIEVE MONTHLY SERIES
# ============================================================

def retrieve_monthly_series(
    query,
    records,
    debug=False
):
    """
    Retrieve the monthly series for one KPI-section pair.

    Example:
        "Show F&B Division revenue for each month"

    Returns January through December for:
        Operating Revenue | F&B Division
    """

    query_clean = rag_normalize(query)

    # --------------------------------------------------------
    # Identify KPI candidates
    # --------------------------------------------------------

    candidate_indices = detect_kpi_candidates(
        query_clean,
        records
    )

    candidate_records = [
        records[i]
        for i in candidate_indices
    ]

    # --------------------------------------------------------
    # Identify section
    # --------------------------------------------------------

    requested_section = detect_requested_section(
        query_clean,
        records
    )

    # --------------------------------------------------------
    # Restrict candidates to requested section
    # --------------------------------------------------------

    if (
        candidate_records
        and requested_section
    ):

        requested_section_clean = (
            rag_normalize(
                requested_section
            )
        )

        filtered = [

            r for r in candidate_records

            if rag_normalize(
                r.get("section", "")
            ) == requested_section_clean
        ]

        if filtered:
            candidate_records = filtered

    # --------------------------------------------------------
    # If no direct KPI match:
    # use cosine similarity as fallback
    # --------------------------------------------------------

    if not candidate_records:

        query_embedding = embedding_model.encode(
            [query_clean],
            convert_to_numpy=True,
            normalize_embeddings=True
        )[0]

        similarity_scores = np.dot(
            rag_embeddings,
            query_embedding
        )

        ranked_indices = np.argsort(
            similarity_scores
        )[::-1]

        # Examine strongest candidates
        for index in ranked_indices[:20]:

            record = records[index]

            # Section restriction if available
            if requested_section:

                if (
                    rag_normalize(
                        record.get("section", "")
                    )
                    != rag_normalize(
                        requested_section
                    )
                ):
                    continue

            candidate_records.append(
                record
            )

    # --------------------------------------------------------
    # Keep ONLY monthly records
    # --------------------------------------------------------

    monthly_records = [

        r for r in candidate_records

        if r.get("period") in MONTH_ORDER
    ]

    # --------------------------------------------------------
    # Sort January → December
    # --------------------------------------------------------

    monthly_records.sort(
        key=lambda r:
            MONTH_ORDER.get(
                r.get("period"),
                999
            )
    )

    # --------------------------------------------------------
    # Remove duplicate month-section-KPI combinations
    # --------------------------------------------------------

    seen = set()

    final_records = []

    for record in monthly_records:

        key = (
            record.get("period"),
            rag_normalize(
                record.get("section", "")
            ),
            rag_normalize(
                record.get("name", "")
            )
        )

        if key not in seen:

            seen.add(key)

            final_records.append(
                record
            )

    # --------------------------------------------------------
    # Debug
    # --------------------------------------------------------

    if debug:

        print("\n" + "=" * 70)
        print("MONTHLY RAG RETRIEVAL")
        print("=" * 70)

        print(
            "Question:",
            query
        )

        print(
            "Detected KPI candidates:",
            len(candidate_records)
        )

        print(
            "Detected section:",
            requested_section
        )

        print(
            "Monthly records:",
            len(final_records)
        )

        print("\nRetrieved records:")

        for record in final_records:

            print(
                f"{record['period']:>10} | "
                f"{record['section']} | "
                f"{record['name']} | "
                f"Actual: {record['actual']:,.2f}"
            )

    return final_records

In [ ]:
result = retrieve_monthly_series(
    "Show F&B Division revenue for each month",
    financial_records,
    debug=True
)

In [ ]:
for r in result:

    print(
        f"{r['period']:>10} | "
        f"Actual = {r['actual']:>15,.2f} | "
        f"Budget = {r['budget']:>15,.2f} | "
        f"Prior = {r['prior']:>15,.2f}"
    )

In [ ]:
monthly_analysis_df = pd.DataFrame([
    {
        "Period": r["period"],
        "Section": r["section"],
        "KPI": r["name"],
        "Actual": r["actual"],
        "Budget": r["budget"],
        "Prior Forecast": r["prior"],
        "Variance to Budget": r["variance_to_budget"],
        "Variance to Budget %": r["variance_to_budget_pct"],
        "Variance to Prior": r["variance_to_prior"],
        "Variance to Prior %": r["variance_to_prior_pct"]
    }
    for r in result
])

monthly_analysis_df

In [ ]:
print("\n" + "=" * 70)
print("F&B OPERATING REVENUE — FULL MONTHLY ANALYSIS DATA")
print("=" * 70)

for r in result:

    print(
        f"{r['period']:>10} | "
        f"Actual: {r['actual']:>14,.2f} | "
        f"Budget: {r['budget']:>14,.2f} | "
        f"Prior: {r['prior']:>14,.2f} | "
        f"Budget Var: {r['variance_to_budget']:>14,.2f} | "
        f"Budget Var %: {r['variance_to_budget_pct']:>8.2f}% | "
        f"Prior Var: {r['variance_to_prior']:>14,.2f} | "
        f"Prior Var %: {r['variance_to_prior_pct']:>8.2f}%"
    )

In [ ]:
# ============================================================
# MONTHLY FINANCIAL ANALYSIS
# ============================================================

def analyze_monthly_series(records):

    if not records:
        return {
            "records": [],
            "summary": {}
        }

    # --------------------------------------------------------
    # Sort chronologically
    # --------------------------------------------------------

    month_order = {
        "January": 1,
        "February": 2,
        "March": 3,
        "April": 4,
        "May": 5,
        "June": 6,
        "July": 7,
        "August": 8,
        "September": 9,
        "October": 10,
        "November": 11,
        "December": 12
    }

    records = sorted(
        records,
        key=lambda r: month_order.get(
            r["period"],
            999
        )
    )

    # --------------------------------------------------------
    # Monthly records
    # --------------------------------------------------------

    analyzed = []

    for record in records:

        analyzed.append({
            "period": record["period"],
            "section": record["section"],
            "name": record["name"],

            "actual": record["actual"],
            "budget": record["budget"],
            "prior": record["prior"],

            "variance_to_budget":
                record["variance_to_budget"],

            "variance_to_budget_pct":
                record["variance_to_budget_pct"],

            "variance_to_prior":
                record["variance_to_prior"],

            "variance_to_prior_pct":
                record["variance_to_prior_pct"]
        })

    # --------------------------------------------------------
    # Summary calculations
    # --------------------------------------------------------

    actual_values = [
        r["actual"]
        for r in records
        if r["actual"] is not None
    ]

    budget_values = [
        r["budget"]
        for r in records
        if r["budget"] is not None
    ]

    prior_values = [
        r["prior"]
        for r in records
        if r["prior"] is not None
    ]

    total_actual = sum(actual_values)
    total_budget = sum(budget_values)
    total_prior = sum(prior_values)

    total_budget_variance = (
        total_actual - total_budget
    )

    total_budget_variance_pct = (
        (
            total_actual - total_budget
        )
        / abs(total_budget)
        * 100
        if total_budget != 0
        else None
    )

    total_prior_variance = (
        total_actual - total_prior
    )

    total_prior_variance_pct = (
        (
            total_actual - total_prior
        )
        / abs(total_prior)
        * 100
        if total_prior != 0
        else None
    )

    # --------------------------------------------------------
    # Highest and lowest actual months
    # --------------------------------------------------------

    valid_actual_records = [
        r for r in records
        if r["actual"] is not None
    ]

    highest_actual = (
        max(
            valid_actual_records,
            key=lambda r: r["actual"]
        )
        if valid_actual_records
        else None
    )

    lowest_actual = (
        min(
            valid_actual_records,
            key=lambda r: r["actual"]
        )
        if valid_actual_records
        else None
    )

    # --------------------------------------------------------
    # Best / worst budget variance percentages
    # --------------------------------------------------------

    valid_budget_variances = [
        r for r in records
        if r["variance_to_budget_pct"] is not None
    ]

    best_budget_month = (
        max(
            valid_budget_variances,
            key=lambda r:
                r["variance_to_budget_pct"]
        )
        if valid_budget_variances
        else None
    )

    worst_budget_month = (
        min(
            valid_budget_variances,
            key=lambda r:
                r["variance_to_budget_pct"]
        )
        if valid_budget_variances
        else None
    )

    # --------------------------------------------------------
    # Months above / below budget
    # --------------------------------------------------------

    above_budget = [
        r for r in records
        if (
            r["variance_to_budget"] is not None
            and r["variance_to_budget"] > 0
        )
    ]

    below_budget = [
        r for r in records
        if (
            r["variance_to_budget"] is not None
            and r["variance_to_budget"] < 0
        )
    ]

    # --------------------------------------------------------
    # Months above / below prior forecast
    # --------------------------------------------------------

    above_prior = [
        r for r in records
        if (
            r["variance_to_prior"] is not None
            and r["variance_to_prior"] > 0
        )
    ]

    below_prior = [
        r for r in records
        if (
            r["variance_to_prior"] is not None
            and r["variance_to_prior"] < 0
        )
    ]

    # --------------------------------------------------------
    # Summary object
    # --------------------------------------------------------

    summary = {

        "number_of_months":
            len(records),

        "total_actual":
            total_actual,

        "total_budget":
            total_budget,

        "total_prior":
            total_prior,

        "total_variance_to_budget":
            total_budget_variance,

        "total_variance_to_budget_pct":
            total_budget_variance_pct,

        "total_variance_to_prior":
            total_prior_variance,

        "total_variance_to_prior_pct":
            total_prior_variance_pct,

        "highest_actual_month":
            highest_actual["period"]
            if highest_actual
            else None,

        "highest_actual_value":
            highest_actual["actual"]
            if highest_actual
            else None,

        "lowest_actual_month":
            lowest_actual["period"]
            if lowest_actual
            else None,

        "lowest_actual_value":
            lowest_actual["actual"]
            if lowest_actual
            else None,

        "best_budget_month":
            best_budget_month["period"]
            if best_budget_month
            else None,

        "best_budget_variance_pct":
            best_budget_month[
                "variance_to_budget_pct"
            ]
            if best_budget_month
            else None,

        "worst_budget_month":
            worst_budget_month["period"]
            if worst_budget_month
            else None,

        "worst_budget_variance_pct":
            worst_budget_month[
                "variance_to_budget_pct"
            ]
            if worst_budget_month
            else None,

        "months_above_budget":
            [
                r["period"]
                for r in above_budget
            ],

        "months_below_budget":
            [
                r["period"]
                for r in below_budget
            ],

        "months_above_prior":
            [
                r["period"]
                for r in above_prior
            ],

        "months_below_prior":
            [
                r["period"]
                for r in below_prior
            ]
    }

    return {
        "records": analyzed,
        "summary": summary
    }

In [ ]:
fnb_monthly_analysis = analyze_monthly_series(
    result
)

print("\n" + "=" * 70)
print("F&B DIVISION MONTHLY ANALYSIS SUMMARY")
print("=" * 70)

for key, value in fnb_monthly_analysis["summary"].items():
    print(f"{key}: {value}")

In [ ]:
# ============================================================
# FINAL AGENT 1 / ask_agent()
# HYBRID SECTION-AWARE FINANCIAL RAG
# ============================================================

import re
import numpy as np
from sentence_transformers import SentenceTransformer


# ============================================================
# 1. EMBEDDING MODEL
# ============================================================

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)


# ============================================================
# 2. TEXT NORMALIZATION
# ============================================================

def rag_normalize(text):

    if text is None:
        return ""

    text = str(text)

    text = text.replace("\xa0", " ")

    text = " ".join(text.split())

    text = text.lower().strip()

    # Common terminology
    text = text.replace(
        "f&b",
        "food and beverage"
    )

    text = text.replace(
        "f & b",
        "food and beverage"
    )

    text = text.replace(
        "food & beverage",
        "food and beverage"
    )

    text = text.replace(
        "adr",
        "average rate"
    )

    text = text.replace(
        "revpar",
        "revenue per available room"
    )

    return text


# ============================================================
# 3. MONTH DEFINITIONS
# ============================================================

MONTH_ORDER = {
    "January": 1,
    "February": 2,
    "March": 3,
    "April": 4,
    "May": 5,
    "June": 6,
    "July": 7,
    "August": 8,
    "September": 9,
    "October": 10,
    "November": 11,
    "December": 12
}


MONTH_ALIASES = {

    "jan": "January",
    "january": "January",

    "feb": "February",
    "february": "February",

    "mar": "March",
    "march": "March",

    "apr": "April",
    "april": "April",

    "may": "May",

    "jun": "June",
    "june": "June",

    "jul": "July",
    "july": "July",

    "aug": "August",
    "august": "August",

    "sep": "September",
    "sept": "September",
    "september": "September",

    "oct": "October",
    "october": "October",

    "nov": "November",
    "november": "November",

    "dec": "December",
    "december": "December"
}


# ============================================================
# 4. DETECT MONTH
# ============================================================

def detect_requested_month(query):

    query_clean = rag_normalize(query)

    for alias, month in MONTH_ALIASES.items():

        if re.search(
            rf"\b{re.escape(alias)}\b",
            query_clean
        ):
            return month

    return None


# ============================================================
# 5. DETECT MONTHLY-ANALYSIS QUESTIONS
# ============================================================

def is_monthly_series_query(query):

    query_clean = rag_normalize(query)

    phrases = [

        "each month",
        "every month",
        "monthly",
        "by month",
        "month by month",
        "across the months",
        "across all months",
        "throughout the year",
        "monthly performance",
        "monthly trend",
        "trend across",
        "12 months",
        "all months",

        # Analytical questions
        "which month",
        "highest month",
        "lowest month",
        "best month",
        "worst month",
        "month performed",
        "monthly analysis"
    ]

    return any(
        phrase in query_clean
        for phrase in phrases
    )


# ============================================================
# 6. SECTION DETECTION
# ============================================================

def detect_requested_section(
    query,
    records
):

    query_clean = rag_normalize(query)

    available_sections = sorted(
        set(
            rag_normalize(
                r.get("section", "")
            )
            for r in records
            if r.get("section")
        ),
        key=len,
        reverse=True
    )

    # --------------------------------------------------------
    # Explicit section
    # --------------------------------------------------------

    for section in available_sections:

        if (
            section
            and section in query_clean
        ):
            return section

    # --------------------------------------------------------
    # Total revenue
    # --------------------------------------------------------

    if "total revenue" in query_clean:

        for section in available_sections:

            if section == "total revenue":
                return section

    # --------------------------------------------------------
    # Operating revenue
    # --------------------------------------------------------

    if "revenue" in query_clean:

        if "total revenue" not in query_clean:

            for section in available_sections:

                if section == "operating revenue":
                    return section

    # --------------------------------------------------------
    # Expenses
    # --------------------------------------------------------

    if (
        "expense" in query_clean
        or "expenses" in query_clean
    ):

        for section in available_sections:

            if "departmental expenses" in section:
                return section

    # --------------------------------------------------------
    # Profit
    # --------------------------------------------------------

    if "profit" in query_clean:

        for section in available_sections:

            if "departmental profit" in section:
                return section

    # --------------------------------------------------------
    # Salaries / wages
    # --------------------------------------------------------

    if (
        "salary" in query_clean
        or "salaries" in query_clean
        or "wage" in query_clean
        or "wages" in query_clean
    ):

        for section in available_sections:

            if (
                "salary" in section
                or "wage" in section
            ):
                return section

    # --------------------------------------------------------
    # Statistics
    # --------------------------------------------------------

    if (
        "occupancy" in query_clean
        or "average rate" in query_clean
        or "revenue per available room" in query_clean
    ):

        for section in available_sections:

            if section == "statistics":
                return section

    return None


# ============================================================
# 7. BUILD SEARCH TEXT FOR EACH RECORD
# ============================================================

def build_rag_document(record):

    name = rag_normalize(
        record.get("name", "")
    )

    section = rag_normalize(
        record.get("section", "")
    )

    period = rag_normalize(
        record.get("period", "")
    )

    extra_terms = []

    # F&B
    if (
        "f&b" in name
        or "food and beverage" in name
    ):

        extra_terms.extend([
            "food",
            "beverage",
            "restaurant",
            "dining"
        ])

    # Rooms
    if "rooms" in name:

        extra_terms.extend([
            "room",
            "hotel rooms",
            "rooms operation"
        ])

    # Revenue
    if "revenue" in section:

        extra_terms.extend([
            "revenue",
            "income",
            "sales"
        ])

    # Expenses
    if "expense" in section:

        extra_terms.extend([
            "expense",
            "expenses",
            "cost",
            "costs"
        ])

    # Profit
    if "profit" in section:

        extra_terms.extend([
            "profit",
            "earnings",
            "gop"
        ])

    # Salaries
    if (
        "salary" in section
        or "wage" in section
    ):

        extra_terms.extend([
            "salary",
            "salaries",
            "wages",
            "payroll",
            "labor"
        ])

    # Statistics
    if section == "statistics":

        extra_terms.extend([
            "occupancy",
            "average rate",
            "revpar",
            "revenue per available room"
        ])

    return " | ".join([
        f"KPI: {name}",
        f"Section: {section}",
        f"Period: {period}",
        "Keywords: " + " ".join(extra_terms)
    ])


# ============================================================
# 8. BUILD EMBEDDING INDEX
# ============================================================

rag_documents = [
    build_rag_document(record)
    for record in financial_records
]

rag_embeddings = embedding_model.encode(
    rag_documents,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print(
    f"RAG index built for "
    f"{len(financial_records)} records."
)


# ============================================================
# 9. FORMAT FINANCIAL VALUES
# ============================================================

def format_value(record, value):

    if value is None:
        return "N/A"

    name = rag_normalize(
        record.get("name", "")
    )

    # Occupancy stored as decimal
    if name == "occupancy":

        return f"{value * 100:.2f}%"

    # All other statistics / financial values
    return f"{value:,.2f}"


# ============================================================
# 10. FIND KPI CANDIDATES
# ============================================================

def find_kpi_candidates(
    query,
    records
):

    query_clean = rag_normalize(query)

    candidates = []

    for index, record in enumerate(records):

        name = rag_normalize(
            record.get("name", "")
        )

        if not name:
            continue

        if name in query_clean:

            candidates.append(index)

    return candidates


# ============================================================
# 11. RETRIEVE ONE RECORD
# ============================================================

def retrieve_best_record(
    query,
    records=financial_records,
    target_period=None,
    default_full_year=False,
    debug=False
):

    query_clean = rag_normalize(query)

    # --------------------------------------------------------
    # Query embedding
    # --------------------------------------------------------

    query_embedding = embedding_model.encode(
        [query_clean],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    cosine_scores = np.dot(
        rag_embeddings,
        query_embedding
    )

    # --------------------------------------------------------
    # Exact KPI candidates
    # --------------------------------------------------------

    exact_candidates = find_kpi_candidates(
        query_clean,
        records
    )

    # --------------------------------------------------------
    # Section
    # --------------------------------------------------------

    requested_section = detect_requested_section(
        query_clean,
        records
    )

    # --------------------------------------------------------
    # Candidate pool
    # --------------------------------------------------------

    if exact_candidates:

        candidate_indices = exact_candidates

    else:

        candidate_indices = list(
            range(len(records))
        )

    # --------------------------------------------------------
    # Restrict to requested period
    # --------------------------------------------------------

    if target_period:

        period_candidates = [
            i
            for i in candidate_indices
            if records[i].get("period")
            == target_period
        ]

        if period_candidates:
            candidate_indices = period_candidates

    # --------------------------------------------------------
    # Otherwise default direct questions to Full-Year
    # --------------------------------------------------------

    elif default_full_year:

        full_year_candidates = [
            i
            for i in candidate_indices
            if records[i].get("period")
            == "Full-Year"
        ]

        if full_year_candidates:
            candidate_indices = (
                full_year_candidates
            )

    # --------------------------------------------------------
    # Score candidates
    # --------------------------------------------------------

    scored = []

    for index in candidate_indices:

        record = records[index]

        name = rag_normalize(
            record.get("name", "")
        )

        section = rag_normalize(
            record.get("section", "")
        )

        period = rag_normalize(
            record.get("period", "")
        )

        score = float(
            cosine_scores[index]
        )

        # ----------------------------------------------------
        # KPI match
        # ----------------------------------------------------

        if name in query_clean:
            score += 0.75

        # ----------------------------------------------------
        # Section match
        # ----------------------------------------------------

        if requested_section:

            requested = rag_normalize(
                requested_section
            )

            if section == requested:

                score += 0.75

        # ----------------------------------------------------
        # Period match
        # ----------------------------------------------------

        if target_period:

            if period == rag_normalize(
                target_period
            ):

                score += 1.00

        # ----------------------------------------------------
        # Revenue
        # ----------------------------------------------------

        if "revenue" in query_clean:

            if (
                "total revenue" in query_clean
                and section == "total revenue"
            ):
                score += 0.50

            elif (
                "total revenue" not in query_clean
                and section == "operating revenue"
            ):
                score += 0.35

        # ----------------------------------------------------
        # Expense
        # ----------------------------------------------------

        if (
            "expense" in query_clean
            or "expenses" in query_clean
        ):

            if "departmental expenses" in section:
                score += 0.45

        # ----------------------------------------------------
        # Profit
        # ----------------------------------------------------

        if "profit" in query_clean:

            if "departmental profit" in section:
                score += 0.45

        # ----------------------------------------------------
        # Salaries/wages
        # ----------------------------------------------------

        if (
            "salary" in query_clean
            or "salaries" in query_clean
            or "wage" in query_clean
            or "wages" in query_clean
        ):

            if (
                "salary" in section
                or "wage" in section
            ):
                score += 0.45

        scored.append({
            "record": record,
            "cosine_similarity": float(
                cosine_scores[index]
            ),
            "final_score": score
        })

    # --------------------------------------------------------
    # No result
    # --------------------------------------------------------

    if not scored:
        return []

    # --------------------------------------------------------
    # Rank
    # --------------------------------------------------------

    scored.sort(
        key=lambda x: x["final_score"],
        reverse=True
    )

    # --------------------------------------------------------
    # Debug
    # --------------------------------------------------------

    if debug:

        print("\n" + "=" * 70)
        print("SINGLE-RECORD RAG DEBUG")
        print("=" * 70)

        print("Question:", query)
        print(
            "Requested period:",
            target_period
        )
        print(
            "Requested section:",
            requested_section
        )

        print("\nTop candidates:")

        for item in scored[:5]:

            r = item["record"]

            print(
                f"\n{r['period']} | "
                f"{r['section']} | "
                f"{r['name']}"
            )

            print(
                f"Cosine: "
                f"{item['cosine_similarity']:.4f}"
            )

            print(
                f"Final score: "
                f"{item['final_score']:.4f}"
            )

    return [
        scored[0]["record"]
    ]


# ============================================================
# 12. IDENTIFY KPI + SECTION FOR MONTHLY SERIES
# ============================================================

def identify_series_target(
    query,
    records=financial_records
):

    # Use semantic retrieval to identify the target
    candidates = retrieve_best_record(
        query,
        records,
        target_period=None,
        default_full_year=False,
        debug=False
    )

    if not candidates:
        return None

    return candidates[0]


# ============================================================
# 13. RETRIEVE ALL 12 MONTHS
# ============================================================

def retrieve_monthly_series(
    query,
    records=financial_records,
    debug=False
):

    target = identify_series_target(
        query,
        records
    )

    if target is None:
        return []

    target_name = rag_normalize(
        target["name"]
    )

    target_section = rag_normalize(
        target["section"]
    )

    # --------------------------------------------------------
    # Collect matching KPI + section across months
    # --------------------------------------------------------

    monthly_records = [

        record

        for record in records

        if record.get("period")
        in MONTH_ORDER

        and rag_normalize(
            record.get("name", "")
        )
        == target_name

        and rag_normalize(
            record.get("section", "")
        )
        == target_section
    ]

    # --------------------------------------------------------
    # Sort chronologically
    # --------------------------------------------------------

    monthly_records.sort(
        key=lambda r:
            MONTH_ORDER.get(
                r["period"],
                999
            )
    )

    # --------------------------------------------------------
    # Remove month duplicates
    # --------------------------------------------------------

    seen = set()
    final_records = []

    for record in monthly_records:

        key = (
            record["period"],
            target_section,
            target_name
        )

        if key not in seen:

            seen.add(key)

            final_records.append(
                record
            )

    if debug:

        print("\n" + "=" * 70)
        print("MONTHLY RAG DEBUG")
        print("=" * 70)

        print(
            "Question:",
            query
        )

        print(
            "Target KPI:",
            target["name"]
        )

        print(
            "Target section:",
            target["section"]
        )

        print(
            "Monthly records:",
            len(final_records)
        )

        for record in final_records:

            print(
                f"{record['period']:>10} | "
                f"{record['section']} | "
                f"{record['name']} | "
                f"Actual: "
                f"{record['actual']:,.2f}"
            )

    return final_records


# ============================================================
# 14. BUILD SINGLE-RECORD CONTEXT
# ============================================================

def build_single_context(record):

    budget_pct = (
        f"{record['variance_to_budget_pct']:.2f}%"
        if record["variance_to_budget_pct"] is not None
        else "N/A"
    )

    prior_pct = (
        f"{record['variance_to_prior_pct']:.2f}%"
        if record["variance_to_prior_pct"] is not None
        else "N/A"
    )

    return f"""
Financial Record

Period: {record['period']}
Section: {record['section']}
KPI: {record['name']}

Actual: {format_value(record, record['actual'])}
Budget: {format_value(record, record['budget'])}
Prior Forecast: {format_value(record, record['prior'])}

Variance to Budget:
{format_value(record, record['variance_to_budget'])}

Variance to Budget %:
{budget_pct}

Variance to Prior Forecast:
{format_value(record, record['variance_to_prior'])}

Variance to Prior Forecast %:
{prior_pct}

Source Sheet: {record['sheet']}
Source Row: {record['source_row']}
""".strip()


# ============================================================
# 15. BUILD MONTHLY CONTEXT
# ============================================================

def build_monthly_context(
    records,
    analysis
):

    lines = []

    if not records:

        return "No monthly financial records were found."

    first = records[0]

    lines.append(
        f"KPI: {first['name']}"
    )

    lines.append(
        f"Section: {first['section']}"
    )

    lines.append("")
    lines.append("MONTHLY RECORDS")
    lines.append("")

    for r in records:

        budget_pct = (
            f"{r['variance_to_budget_pct']:.2f}%"
            if r["variance_to_budget_pct"]
            is not None
            else "N/A"
        )

        prior_pct = (
            f"{r['variance_to_prior_pct']:.2f}%"
            if r["variance_to_prior_pct"]
            is not None
            else "N/A"
        )

        lines.append(
            f"""
Period: {r['period']}
Actual: {format_value(r, r['actual'])}
Budget: {format_value(r, r['budget'])}
Prior Forecast: {format_value(r, r['prior'])}
Variance to Budget: {format_value(r, r['variance_to_budget'])}
Variance to Budget %: {budget_pct}
Variance to Prior: {format_value(r, r['variance_to_prior'])}
Variance to Prior %: {prior_pct}
""".strip()
        )

    summary = analysis["summary"]

    lines.append("")
    lines.append("AGGREGATED ANALYSIS")
    lines.append("")

    lines.append(
        f"Total Actual: "
        f"{summary['total_actual']:,.2f}"
    )

    lines.append(
        f"Total Budget: "
        f"{summary['total_budget']:,.2f}"
    )

    lines.append(
        f"Total Prior Forecast: "
        f"{summary['total_prior']:,.2f}"
    )

    lines.append(
        f"Total Variance to Budget: "
        f"{summary['total_variance_to_budget']:,.2f}"
    )

    lines.append(
        f"Total Variance to Budget %: "
        f"{summary['total_variance_to_budget_pct']:.2f}%"
    )

    lines.append(
        f"Total Variance to Prior: "
        f"{summary['total_variance_to_prior']:,.2f}"
    )

    lines.append(
        f"Total Variance to Prior %: "
        f"{summary['total_variance_to_prior_pct']:.2f}%"
    )

    lines.append(
        f"Highest Actual Month: "
        f"{summary['highest_actual_month']} "
        f"({summary['highest_actual_value']:,.2f})"
    )

    lines.append(
        f"Lowest Actual Month: "
        f"{summary['lowest_actual_month']} "
        f"({summary['lowest_actual_value']:,.2f})"
    )

    lines.append(
        f"Best Budget Variance Month: "
        f"{summary['best_budget_month']} "
        f"({summary['best_budget_variance_pct']:.2f}%)"
    )

    lines.append(
        f"Worst Budget Variance Month: "
        f"{summary['worst_budget_month']} "
        f"({summary['worst_budget_variance_pct']:.2f}%)"
    )

    lines.append(
        "Months Above Budget: "
        + ", ".join(
            summary["months_above_budget"]
        )
    )

    lines.append(
        "Months Below Budget: "
        + ", ".join(
            summary["months_below_budget"]
        )
    )

    lines.append(
        "Months Above Prior Forecast: "
        + ", ".join(
            summary["months_above_prior"]
        )
    )

    lines.append(
        "Months Below Prior Forecast: "
        + ", ".join(
            summary["months_below_prior"]
        )
    )

    return "\n".join(lines)


# ============================================================
# 16. DETERMINE ANSWER MODE
# ============================================================

def determine_rag_mode(query):

    requested_month = detect_requested_month(
        query
    )

    monthly_analysis = (
        is_monthly_series_query(query)
    )

    # Explicit month has priority over generic
    # "monthly" wording.
    if (
        requested_month is not None
        and not (
            monthly_analysis
            and (
                "each month" in rag_normalize(query)
                or "all months" in rag_normalize(query)
                or "every month" in rag_normalize(query)
                or "12 months" in rag_normalize(query)
            )
        )
    ):
        return "SINGLE", requested_month

    if monthly_analysis:
        return "MONTHLY_SERIES", None

    # Direct questions default to Full-Year
    return "SINGLE", "Full-Year"


# ============================================================
# 17. FINAL ask_agent()
# ============================================================

def ask_agent(
    query,
    debug=False
):
    """
    Main Agent 1 function.

    Modes:
        SINGLE
            Retrieves one relevant record.

        MONTHLY_SERIES
            Retrieves the complete January-December
            series for one KPI-section pair.
    """

    query = str(query).strip()

    if not query:

        return "Please provide a financial question."

    # --------------------------------------------------------
    # Determine retrieval mode
    # --------------------------------------------------------

    mode, period = determine_rag_mode(
        query
    )

    if debug:

        print("\n" + "=" * 70)
        print("AGENT 1")
        print("=" * 70)

        print("Question:", query)
        print("Mode:", mode)

        if period:
            print("Period:", period)

    # ========================================================
    # MODE 1: MONTHLY SERIES
    # ========================================================

    if mode == "MONTHLY_SERIES":

        monthly_records = retrieve_monthly_series(
            query,
            financial_records,
            debug=debug
        )

        if not monthly_records:

            return (
                "I could not find the requested KPI and "
                "section across the monthly financial data."
            )

        # ----------------------------------------------------
        # Analyze monthly series
        # ----------------------------------------------------

        analysis = analyze_monthly_series(
            monthly_records
        )

        # ----------------------------------------------------
        # Build grounded context
        # ----------------------------------------------------

        context = build_monthly_context(
            monthly_records,
            analysis
        )

        # ----------------------------------------------------
        # Generation prompt
        # ----------------------------------------------------

        prompt = f"""
You are a hotel financial analysis agent.

Answer the user's question using ONLY the supplied
financial context.

Do not invent figures.

Do not introduce information that is not present
in the context.

The financial data has already been extracted and
the variances and aggregate calculations have
already been calculated.

Your task is to INTERPRET the supplied results.

Pay attention to:
- Actual
- Budget
- Prior Forecast
- Monthly trends
- Budget variances
- Prior Forecast variances
- Highest and lowest months
- Months above or below budget

Use clear Markdown.

For a monthly analysis, use:

# Financial Insights

## Key Financial Metrics

| Month | Actual | Budget | Prior |
|---|---:|---:|---:|

## Analysis

Explain the overall performance and the important
monthly movements.

USER QUESTION:
{query}

FINANCIAL CONTEXT:
{context}
"""

    # ========================================================
    # MODE 2: SINGLE RECORD
    # ========================================================

    else:

        retrieved = retrieve_best_record(
            query,
            financial_records,
            target_period=period,
            default_full_year=True,
            debug=debug
        )

        if not retrieved:

            return (
                "I could not find a matching financial "
                "record in the available data."
            )

        record = retrieved[0]

        # ----------------------------------------------------
        # Build context
        # ----------------------------------------------------

        context = build_single_context(
            record
        )

        # ----------------------------------------------------
        # Generation prompt
        # ----------------------------------------------------

        prompt = f"""
You are a hotel financial analysis agent.

Answer the user's question using ONLY the supplied
financial context.

Do not invent financial figures.

Do not change Actual, Budget, or Prior Forecast.

The numerical calculations have already been
performed by the system.

Use the supplied variance values when discussing
comparisons.

Use clear Markdown.

Format the answer as:

# Executive Summary

# Financial Insights

## Key Financial Metrics

| KPI | Actual | Budget | Prior |
|---|---:|---:|---:|

## Analysis

Explain the result.

When a budget comparison is requested, state:
- the variance amount
- the variance percentage
- whether Actual was above or below Budget

When a prior forecast comparison is requested,
do the equivalent.

USER QUESTION:
{query}

RETRIEVED FINANCIAL CONTEXT:
{context}
"""

    # ========================================================
    # LLM GENERATION
    # ========================================================

    try:

        response = client.chat.completions.create(

            model="openai/gpt-oss-20b",

            messages=[

                {
                    "role": "system",
                    "content": (
                        "You are a precise hotel financial "
                        "analysis assistant. "
                        "Use only supplied financial data."
                    )
                },

                {
                    "role": "user",
                    "content": prompt
                }
            ],

            temperature=0
        )

        answer = (
            response
            .choices[0]
            .message
            .content
            .strip()
        )

        return answer

    except Exception as e:

        print(
            "LLM generation error:",
            e
        )

        return (
            "The financial record was retrieved successfully, "
            "but the final response could not be generated."
        )

In [ ]:
print(
    ask_agent(
        "What was the total revenue?",
        debug=True
    )
)

In [ ]:
print(
    ask_agent(
        "What was F&B Division revenue in March?",
        debug=True
    )
)

In [ ]:
print(
    ask_agent(
        "How did F&B Division, Departmental Expenses perform across the 12 months?",
        debug=True
    )
)

In [ ]:
print(
    ask_agent(
        "How did RevPAR, statistics perform across the 12 months?",
        debug=True
    )
)

In [ ]:
print(
    ask_agent(
        "Which month had the highest F&B Division revenue?",
        debug=True
    )
)

In [ ]:
# ============================================================
# MASTER ROUTER FOR AGENT 1
# ============================================================

def route_financial_query(query):
    """
    Determine the appropriate route for a financial question.

    Routes:
        RETRIEVAL
        VARIANCE
        COMPARISON
        DIAGNOSTIC
    """

    q = rag_normalize(query)

    # --------------------------------------------------------
    # Diagnostic / why questions
    # --------------------------------------------------------

    diagnostic_terms = [
        "why",
        "reason",
        "cause",
        "causes",
        "driver",
        "drivers",
        "underperforming",
        "underperform",
        "what caused",
        "explain the decline",
        "explain the decrease",
        "explain the increase",
        "problem",
        "issue"
    ]

    if any(
        term in q
        for term in diagnostic_terms
    ):
        return "DIAGNOSTIC"

    # --------------------------------------------------------
    # Comparison questions
    # --------------------------------------------------------

    comparison_terms = [
        "compare",
        "comparison",
        "versus",
        "vs",
        "difference between",
        "higher than",
        "lower than",
        "highest",
        "lowest",
        "greater than",
        "less than",
        "which was highest",
        "which was lowest"
    ]

    if any(
        term in q
        for term in comparison_terms
    ):
        return "COMPARISON"

    # --------------------------------------------------------
    # Variance questions
    # --------------------------------------------------------

    variance_terms = [
        "variance",
        "budget",
        "budgeted",
        "below budget",
        "above budget",
        "against budget",
        "prior forecast",
        "compared with prior",
        "compared to prior",
        "over budget",
        "under budget"
    ]

    if any(
        term in q
        for term in variance_terms
    ):
        return "VARIANCE"

    # --------------------------------------------------------
    # Default
    # --------------------------------------------------------

    return "RETRIEVAL"

In [ ]:
# ============================================================
# COMPARISON RECORD RETRIEVAL
# ============================================================

def retrieve_comparison_records(
    query,
    records=financial_records,
    max_records=10,
    debug=False
):
    """
    Retrieve a related set of records for comparison.

    Examples:
        Which average rate was highest?
        Compare Rooms Division and F&B Division revenue.
    """

    q = rag_normalize(query)

    # --------------------------------------------------------
    # Identify section
    # --------------------------------------------------------

    section = detect_requested_section(
        q,
        records
    )

    # --------------------------------------------------------
    # Identify explicitly mentioned KPIs
    # --------------------------------------------------------

    mentioned = []

    for record in records:

        name = rag_normalize(
            record.get("name", "")
        )

        if (
            name
            and name in q
            and name not in mentioned
        ):
            mentioned.append(name)

    # --------------------------------------------------------
    # Special case: Average Rate comparison
    # --------------------------------------------------------

    if "average rate" in q:

        candidates = [

            r for r in records

            if (
                "average rate"
                in rag_normalize(
                    r.get("name", "")
                )
            )

            and (
                section is None
                or rag_normalize(
                    r.get("section", "")
                )
                == rag_normalize(section)
                or rag_normalize(
                    r.get("section", "")
                )
                == "statistics"
            )

            and r.get("period") == "Full-Year"
        ]

        # If Full-Year restriction produces nothing,
        # search all records.
        if not candidates:

            candidates = [

                r for r in records

                if "average rate"
                in rag_normalize(
                    r.get("name", "")
                )
            ]

        # Remove duplicate KPI names within this comparison
        seen = set()
        result = []

        for r in candidates:

            name = rag_normalize(
                r["name"]
            )

            if name not in seen:

                seen.add(name)
                result.append(r)

        if debug:

            print("\n" + "=" * 70)
            print("COMPARISON RETRIEVAL")
            print("=" * 70)

            print("Question:", query)

            for r in result:

                print(
                    f"{r['section']} | "
                    f"{r['name']} | "
                    f"Actual: {r['actual']:,.2f}"
                )

        return result[:max_records]

    # --------------------------------------------------------
    # Explicit KPI comparison
    # --------------------------------------------------------

    if len(mentioned) >= 2:

        result = []

        for record in records:

            name = rag_normalize(
                record.get("name", "")
            )

            record_section = rag_normalize(
                record.get("section", "")
            )

            if name in mentioned:

                if section is None or (
                    record_section
                    == rag_normalize(section)
                ):

                    # Prefer Full-Year
                    if record.get(
                        "period"
                    ) == "Full-Year":

                        result.append(record)

        if result:
            return result[:max_records]

    # --------------------------------------------------------
    # Semantic fallback
    # --------------------------------------------------------

    query_embedding = embedding_model.encode(
        [q],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    similarity = np.dot(
        rag_embeddings,
        query_embedding
    )

    indices = np.argsort(
        similarity
    )[::-1]

    result = []

    seen = set()

    for index in indices:

        record = records[index]

        # Prefer Full-Year for direct comparisons
        if record.get("period") != "Full-Year":
            continue

        name = rag_normalize(
            record.get("name", "")
        )

        if name in seen:
            continue

        seen.add(name)

        result.append(record)

        if len(result) >= max_records:
            break

    return result

In [ ]:
# ============================================================
# COMPARISON ANALYSIS
# ============================================================

def analyze_comparison(
    records,
    query
):

    if not records:
        return {
            "records": [],
            "highest": None,
            "lowest": None,
            "direction": None
        }

    valid = [

        r for r in records

        if r.get("actual") is not None
    ]

    if not valid:

        return {
            "records": records,
            "highest": None,
            "lowest": None,
            "direction": None
        }

    q = rag_normalize(query)

    highest = max(
        valid,
        key=lambda r: r["actual"]
    )

    lowest = min(
        valid,
        key=lambda r: r["actual"]
    )

    direction = None

    if "lowest" in q:
        direction = "lowest"

    elif "highest" in q:
        direction = "highest"

    return {
        "records": valid,
        "highest": highest,
        "lowest": lowest,
        "direction": direction
    }

In [ ]:
def build_comparison_context(
    analysis
):

    records = analysis["records"]

    lines = []

    for r in records:

        lines.append(
            f"""
Period: {r['period']}
Section: {r['section']}
KPI: {r['name']}
Actual: {format_value(r, r['actual'])}
Budget: {format_value(r, r['budget'])}
Prior Forecast: {format_value(r, r['prior'])}
Variance to Budget: {format_value(r, r['variance_to_budget'])}
Variance to Budget %: {
    f"{r['variance_to_budget_pct']:.2f}%"
    if r['variance_to_budget_pct'] is not None
    else "N/A"
}
Variance to Prior: {format_value(r, r['variance_to_prior'])}
Variance to Prior %: {
    f"{r['variance_to_prior_pct']:.2f}%"
    if r['variance_to_prior_pct'] is not None
    else "N/A"
}
""".strip()
        )

    if analysis["highest"]:

        r = analysis["highest"]

        lines.append(
            "\nHIGHEST ACTUAL:\n"
            f"{r['name']} = "
            f"{format_value(r, r['actual'])}"
        )

    if analysis["lowest"]:

        r = analysis["lowest"]

        lines.append(
            "\nLOWEST ACTUAL:\n"
            f"{r['name']} = "
            f"{format_value(r, r['actual'])}"
        )

    return "\n\n".join(lines)

In [ ]:
# ============================================================
# DIAGNOSTIC AGENT
# ============================================================

def diagnostic_agent(
    query,
    max_iterations=2,
    debug=False
):
    """
    Diagnostic reasoning loop.

    First attempt:
        retrieve the most relevant record.

    If the query requires broader context:
        retrieve the monthly series.

    Maximum iterations prevent endless loops.
    """

    if debug:

        print("\n" + "=" * 70)
        print("DIAGNOSTIC LOOP")
        print("=" * 70)

        print(
            "Maximum iterations:",
            max_iterations
        )

    # --------------------------------------------------------
    # Iteration 1: single-record retrieval
    # --------------------------------------------------------

    record_result = retrieve_best_record(
        query,
        financial_records,
        target_period=None,
        default_full_year=True,
        debug=debug
    )

    if record_result:

        record = record_result[0]

        # ----------------------------------------------------
        # Check whether broader context is needed
        # ----------------------------------------------------

        q = rag_normalize(query)

        needs_monthly_context = any(
            term in q
            for term in [
                "trend",
                "throughout the year",
                "across the year",
                "monthly",
                "over the months",
                "over time"
            ]
        )

        if needs_monthly_context:

            monthly = retrieve_monthly_series(
                query,
                financial_records,
                debug=debug
            )

            if monthly:

                analysis = analyze_monthly_series(
                    monthly
                )

                return {
                    "mode": "MONTHLY_SERIES",
                    "records": monthly,
                    "analysis": analysis
                }

        # ----------------------------------------------------
        # Single record is sufficient
        # ----------------------------------------------------

        return {
            "mode": "SINGLE",
            "records": record_result,
            "analysis": None
        }

    # --------------------------------------------------------
    # Iteration 2: broaden to monthly series
    # --------------------------------------------------------

    if max_iterations > 1:

        monthly = retrieve_monthly_series(
            query,
            financial_records,
            debug=debug
        )

        if monthly:

            analysis = analyze_monthly_series(
                monthly
            )

            return {
                "mode": "MONTHLY_SERIES",
                "records": monthly,
                "analysis": analysis
            }

    return {
        "mode": "NONE",
        "records": [],
        "analysis": None
    }

In [ ]:
# ============================================================
# MULTI-DIMENSIONAL FINANCIAL ANALYSIS
# ============================================================
#
# This layer allows Agent 1 to analyze:
#
# 1. Information within a section
# 2. Information across sections
# 3. Information across months
# 4. Information across sections AND months
#
# Python performs the numerical calculations.
# The LLM is used only to explain the verified results.
# ============================================================

import re
import numpy as np
import pandas as pd
from IPython.display import display, HTML


# ============================================================
# 1. MONTH ORDER
# ============================================================

MONTH_ORDER = {
    "January": 1,
    "February": 2,
    "March": 3,
    "April": 4,
    "May": 5,
    "June": 6,
    "July": 7,
    "August": 8,
    "September": 9,
    "October": 10,
    "November": 11,
    "December": 12
}


# ============================================================
# 2. SAFE NUMERIC CONVERSION
# ============================================================

def md_safe_float(value):

    if value is None:
        return None

    try:

        if pd.isna(value):
            return None

    except Exception:
        pass

    try:
        return float(value)

    except (TypeError, ValueError):
        return None


# ============================================================
# 3. FORMAT NUMBER
# ============================================================

def md_format_number(value):

    value = md_safe_float(value)

    if value is None:
        return "N/A"

    return f"{value:,.2f}"


# ============================================================
# 4. FORMAT PERCENTAGE
# ============================================================

def md_format_pct(value):

    value = md_safe_float(value)

    if value is None:
        return "N/A"

    return f"{value:.2f}%"


# ============================================================
# 5. NORMALIZE SEARCH TEXT
# ============================================================

def md_normalize(text):

    if text is None:
        return ""

    text = str(text).lower()

    text = (
        text
        .replace("&", " and ")
        .replace("-", " ")
        .replace("_", " ")
        .replace("/", " ")
    )

    text = re.sub(
        r"[^a-z0-9\s]",
        " ",
        text
    )

    text = " ".join(
        text.split()
    )

    return text


# ============================================================
# 6. CALCULATE VARIANCE
# ============================================================

def md_variance(actual, comparison):

    actual = md_safe_float(actual)
    comparison = md_safe_float(comparison)

    if (
        actual is None
        or comparison is None
    ):
        return None

    return actual - comparison


def md_variance_pct(actual, comparison):

    actual = md_safe_float(actual)
    comparison = md_safe_float(comparison)

    if (
        actual is None
        or comparison is None
        or comparison == 0
    ):
        return None

    return (
        (actual - comparison)
        / abs(comparison)
    ) * 100


# ============================================================
# 7. MONTH DETECTION
# ============================================================

def md_detect_month(query):

    q = md_normalize(query)

    for month in MONTH_ORDER:

        if month.lower() in q:
            return month

    return None


# ============================================================
# 8. DETERMINE WHETHER QUESTION IS MONTHLY
# ============================================================

def md_is_monthly_question(query):

    q = md_normalize(query)

    monthly_terms = [

        "monthly",
        "month",
        "months",
        "across the year",
        "throughout the year",
        "over the year",
        "year",
        "from january",
        "from february",
        "from march",
        "from april",
        "from may",
        "from june",
        "from july",
        "from august",
        "from september",
        "from october",
        "from november",
        "from december",
        "each month",
        "every month",
        "month by month",
        "monthly trend",
        "trend across"
    ]

    return any(
        term in q
        for term in monthly_terms
    )


# ============================================================
# 9. DETERMINE ANALYSIS DIMENSION
# ============================================================

def md_determine_analysis_type(query):

    q = md_normalize(query)

    monthly = md_is_monthly_question(query)

    cross_section_terms = [

        "across sections",
        "between sections",
        "among sections",
        "across departments",
        "between departments",
        "contribute to",
        "contribution to",
        "affect",
        "affected",
        "impact",
        "relationship",
        "relate",
        "related",
        "factors",
        "drivers",
        "what caused",
        "why did",
        "why was",
        "profitability",
        "performance of revenue and",
        "revenue and profit",
        "revenue,",
        "expenses and",
        "revenue and expenses"
    ]

    cross_section = any(
        term in q
        for term in cross_section_terms
    )

    if monthly and cross_section:

        return "CROSS_SECTION_MONTH"

    if monthly:

        return "MONTHLY"

    if cross_section:

        return "CROSS_SECTION"

    return "WITHIN_SECTION"


# ============================================================
# 10. KPI ALIASES
# ============================================================
#
# These aliases help the agent identify the relevant
# financial records in the Excel workbook.
# ============================================================

KPI_ALIASES = {

    "revenue": [
        "total revenue",
        "rooms division",
        "f&b division",
        "other operating division",
        "miscellaneous income",
        "rooms revenue",
        "food revenue",
        "beverage revenue"
    ],

    "rooms": [
        "occupancy",
        "rooms available",
        "rooms sold",
        "average rate",
        "revpar",
        "rooms revenue",
        "transient rooms",
        "group rooms",
        "contract rooms"
    ],

    "fb": [
        "f&b division",
        "food and beverage",
        "f&b",
        "food revenue",
        "beverage revenue",
        "average check",
        "customers",
        "food cost",
        "beverage cost"
    ],

    "profit": [
        "departmental profit",
        "gross operating profit",
        "gop",
        "ebitda",
        "total profit"
    ],

    "expenses": [
        "departmental expenses",
        "total departmental expenses",
        "undistributed expenses",
        "total undistributed expenses",
        "management fees",
        "ffe replacement reserves",
        "expense"
    ],

    "profitability_chain": [
        "total revenue",
        "total departmental expenses",
        "total departmental profit",
        "total undistributed expenses",
        "gross operating profit",
        "management fees",
        "ebitda",
        "ffe replacement reserves",
        "total profit"
    ]
}


# ============================================================
# 11. IDENTIFY RELEVANT KPI GROUPS
# ============================================================

def md_identify_groups(query):

    q = md_normalize(query)

    groups = []

    if any(
        term in q
        for term in [
            "room",
            "occupancy",
            "revpar",
            "average rate"
        ]
    ):
        groups.append("rooms")

    if any(
        term in q
        for term in [
            "f&b",
            "food",
            "beverage",
            "banquet",
            "average check",
            "customer"
        ]
    ):
        groups.append("fb")

    if any(
        term in q
        for term in [
            "revenue",
            "income"
        ]
    ):
        groups.append("revenue")

    if any(
        term in q
        for term in [
            "profit",
            "gop",
            "ebitda"
        ]
    ):
        groups.append("profit")

    if any(
        term in q
        for term in [
            "expense",
            "expenses",
            "cost",
            "management fee",
            "reserve"
        ]
    ):
        groups.append("expenses")

    # --------------------------------------------------------
    # If question asks about overall performance,
    # use the profitability chain.
    # --------------------------------------------------------

    if any(
        term in q
        for term in [
            "overall performance",
            "financial performance",
            "overall financial",
            "financial position",
            "comprehensive analysis",
            "overall analysis"
        ]
    ):

        groups.append(
            "profitability_chain"
        )

    # Remove duplicates
    groups = list(
        dict.fromkeys(groups)
    )

    return groups


# ============================================================
# 12. BUILD KPI TARGET LIST
# ============================================================

def md_build_kpi_targets(
    query
):

    groups = md_identify_groups(
        query
    )

    targets = []

    for group in groups:

        targets.extend(
            KPI_ALIASES.get(
                group,
                []
            )
        )

    # If no specific group is identified,
    # retrieve important financial KPIs.

    if not targets:

        targets = [

            "total revenue",
            "rooms division",
            "f&b division",
            "gross operating profit",
            "ebitda",
            "total profit"
        ]

    # Preserve order while removing duplicates

    targets = list(
        dict.fromkeys(
            targets
        )
    )

    return targets


# ============================================================
# 13. MATCH RECORD TO TARGET
# ============================================================

def md_record_matches_target(
    record,
    targets
):

    name = md_normalize(
        record.get(
            "name",
            ""
        )
    )

    section = md_normalize(
        record.get(
            "section",
            ""
        )
    )

    combined = (
        name + " " + section
    )

    for target in targets:

        target_normalized = md_normalize(
            target
        )

        if (
            target_normalized in name
            or target_normalized in combined
        ):

            return True

    return False


# ============================================================
# 14. REMOVE DUPLICATES
# ============================================================

def md_deduplicate_records(
    records
):

    unique = {}

    for record in records:

        key = (

            record.get(
                "period"
            ),

            md_normalize(
                record.get(
                    "section",
                    ""
                )
            ),

            md_normalize(
                record.get(
                    "name",
                    ""
                )
            )
        )

        if key not in unique:

            unique[key] = record

    return list(
        unique.values()
    )


# ============================================================
# 15. RETRIEVE MULTI-DIMENSIONAL RECORDS
# ============================================================

def retrieve_multidimensional_records(
    query,
    records=financial_records,
    debug=False
):

    analysis_type = (
        md_determine_analysis_type(
            query
        )
    )

    targets = md_build_kpi_targets(
        query
    )

    requested_month = md_detect_month(
        query
    )

    selected = []

    # ========================================================
    # FULL-YEAR / SPECIFIC MONTH
    # ========================================================

    if analysis_type in [
        "WITHIN_SECTION",
        "CROSS_SECTION"
    ]:

        target_period = (
            requested_month
            if requested_month
            else "Full-Year"
        )

        for record in records:

            if record.get(
                "period"
            ) != target_period:

                continue

            if md_record_matches_target(
                record,
                targets
            ):

                selected.append(
                    record
                )

    # ========================================================
    # MONTHLY / CROSS-SECTION + MONTH
    # ========================================================

    else:

        for record in records:

            period = record.get(
                "period"
            )

            if period not in MONTH_ORDER:
                continue

            if md_record_matches_target(
                record,
                targets
            ):

                selected.append(
                    record
                )

    # ========================================================
    # DEDUPLICATE
    # ========================================================

    selected = md_deduplicate_records(
        selected
    )

    # ========================================================
    # SORT
    # ========================================================

    selected.sort(

        key=lambda r: (

            MONTH_ORDER.get(
                r.get("period"),
                0
            ),

            md_normalize(
                r.get(
                    "section",
                    ""
                )
            ),

            md_normalize(
                r.get(
                    "name",
                    ""
                )
            )
        )
    )

    # ========================================================
    # DEBUG
    # ========================================================

    if debug:

        print("\n" + "=" * 80)
        print("MULTI-DIMENSIONAL RETRIEVAL")
        print("=" * 80)

        print(
            "Analysis Type:",
            analysis_type
        )

        print(
            "Targets:",
            targets
        )

        print(
            "Requested Month:",
            requested_month
        )

        print(
            "Records Retrieved:",
            len(selected)
        )

        print("-" * 80)

        for record in selected:

            print(

                f"{record.get('period')} | "
                f"{record.get('section')} | "
                f"{record.get('name')} | "
                f"Actual: "
                f"{md_format_number(record.get('actual'))}"
            )

    return selected


# ============================================================
# 16. BUILD UNIVERSAL DATAFRAME
# ============================================================

def build_multidimensional_dataframe(
    records
):

    rows = []

    for record in records:

        actual = md_safe_float(
            record.get("actual")
        )

        budget = md_safe_float(
            record.get("budget")
        )

        prior = md_safe_float(
            record.get("prior")
        )

        budget_variance = (
            md_variance(
                actual,
                budget
            )
        )

        budget_variance_pct = (
            md_variance_pct(
                actual,
                budget
            )
        )

        prior_variance = (
            md_variance(
                actual,
                prior
            )
        )

        prior_variance_pct = (
            md_variance_pct(
                actual,
                prior
            )
        )

        rows.append({

            "Period":
                record.get("period"),

            "Section":
                record.get("section"),

            "KPI":
                record.get("name"),

            "Actual":
                actual,

            "Budget":
                budget,

            "Prior Forecast":
                prior,

            "Budget Variance":
                budget_variance,

            "Budget Variance %":
                budget_variance_pct,

            "Prior Variance":
                prior_variance,

            "Prior Variance %":
                prior_variance_pct
        })

    df = pd.DataFrame(
        rows
    )

    if df.empty:
        return df

    # --------------------------------------------------------
    # Sort
    # --------------------------------------------------------

    df["_month_order"] = (
        df["Period"]
        .map(MONTH_ORDER)
        .fillna(0)
    )

    df = df.sort_values(
        [
            "_month_order",
            "Section",
            "KPI"
        ]
    )

    df = df.drop(
        columns=[
            "_month_order"
        ]
    )

    df = df.reset_index(
        drop=True
    )

    return df


# ============================================================
# 17. ANALYZE MONTHLY DATA
# ============================================================

def analyze_multidimensional_monthly(
    records
):

    if not records:

        return {
            "records": [],
            "summary": {}
        }

    # --------------------------------------------------------
    # Group by KPI
    # --------------------------------------------------------

    groups = {}

    for record in records:

        key = (

            record.get(
                "section"
            ),

            record.get(
                "name"
            )
        )

        groups.setdefault(
            key,
            []
        ).append(
            record
        )

    kpi_summaries = []

    # ========================================================
    # ANALYZE EACH KPI
    # ========================================================

    for key, kpi_records in groups.items():

        section, name = key

        kpi_records.sort(

            key=lambda r:
                MONTH_ORDER.get(
                    r.get("period"),
                    999
                )
        )

        actuals = [
            md_safe_float(
                r.get("actual")
            )
            for r in kpi_records
            if md_safe_float(
                r.get("actual")
            ) is not None
        ]

        budgets = [
            md_safe_float(
                r.get("budget")
            )
            for r in kpi_records
            if md_safe_float(
                r.get("budget")
            ) is not None
        ]

        priors = [
            md_safe_float(
                r.get("prior")
            )
            for r in kpi_records
            if md_safe_float(
                r.get("prior")
            ) is not None
        ]

        if not actuals:
            continue

        total_actual = sum(
            actuals
        )

        total_budget = (
            sum(budgets)
            if budgets
            else None
        )

        total_prior = (
            sum(priors)
            if priors
            else None
        )

        highest = max(
            [
                r for r in kpi_records
                if r.get("actual") is not None
            ],
            key=lambda r:
                r.get("actual")
        )

        lowest = min(
            [
                r for r in kpi_records
                if r.get("actual") is not None
            ],
            key=lambda r:
                r.get("actual")
        )

        budget_variance = (
            md_variance(
                total_actual,
                total_budget
            )
        )

        budget_variance_pct = (
            md_variance_pct(
                total_actual,
                total_budget
            )
        )

        prior_variance = (
            md_variance(
                total_actual,
                total_prior
            )
        )

        prior_variance_pct = (
            md_variance_pct(
                total_actual,
                total_prior
            )
        )

        above_budget = [

            r for r in kpi_records

            if (
                r.get(
                    "variance_to_budget"
                ) is not None

                and r.get(
                    "variance_to_budget"
                ) > 0
            )
        ]

        below_budget = [

            r for r in kpi_records

            if (
                r.get(
                    "variance_to_budget"
                ) is not None

                and r.get(
                    "variance_to_budget"
                ) < 0
            )
        ]

        kpi_summaries.append({

            "section":
                section,

            "kpi":
                name,

            "months":
                len(kpi_records),

            "total_actual":
                total_actual,

            "total_budget":
                total_budget,

            "total_prior":
                total_prior,

            "budget_variance":
                budget_variance,

            "budget_variance_pct":
                budget_variance_pct,

            "prior_variance":
                prior_variance,

            "prior_variance_pct":
                prior_variance_pct,

            "highest_month":
                highest.get("period"),

            "highest_value":
                highest.get("actual"),

            "lowest_month":
                lowest.get("period"),

            "lowest_value":
                lowest.get("actual"),

            "months_above_budget":
                [
                    r.get("period")
                    for r in above_budget
                ],

            "months_below_budget":
                [
                    r.get("period")
                    for r in below_budget
                ]
        })

    return {

        "records":
            records,

        "kpi_summaries":
            kpi_summaries
    }


# ============================================================
# 18. BUILD ANALYSIS CONTEXT FOR LLM
# ============================================================

def build_multidimensional_context(
    records,
    analysis_type,
    analysis
):

    lines = []

    lines.append(
        f"Analysis type: {analysis_type}"
    )

    lines.append(
        f"Number of verified records: {len(records)}"
    )

    lines.append("")

    lines.append(
        "VERIFIED FINANCIAL RECORDS:"
    )

    for record in records:

        lines.append(

            f"Period={record.get('period')} | "
            f"Section={record.get('section')} | "
            f"KPI={record.get('name')} | "
            f"Actual={md_format_number(record.get('actual'))} | "
            f"Budget={md_format_number(record.get('budget'))} | "
            f"Prior Forecast={md_format_number(record.get('prior'))} | "
            f"Budget Variance={md_format_number(record.get('variance_to_budget'))} | "
            f"Budget Variance %={md_format_pct(record.get('variance_to_budget_pct'))} | "
            f"Prior Variance={md_format_number(record.get('variance_to_prior'))} | "
            f"Prior Variance %={md_format_pct(record.get('variance_to_prior_pct'))}"
        )

    # ========================================================
    # MONTHLY SUMMARY
    # ========================================================

    if analysis:

        kpi_summaries = analysis.get(
            "kpi_summaries",
            []
        )

        if kpi_summaries:

            lines.append("")

            lines.append(
                "CALCULATED KPI SUMMARIES:"
            )

            for item in kpi_summaries:

                lines.append(

                    f"Section={item['section']} | "
                    f"KPI={item['kpi']} | "
                    f"Total Actual={md_format_number(item['total_actual'])} | "
                    f"Total Budget={md_format_number(item['total_budget'])} | "
                    f"Total Prior={md_format_number(item['total_prior'])} | "
                    f"Budget Variance={md_format_number(item['budget_variance'])} | "
                    f"Budget Variance %={md_format_pct(item['budget_variance_pct'])} | "
                    f"Prior Variance={md_format_number(item['prior_variance'])} | "
                    f"Prior Variance %={md_format_pct(item['prior_variance_pct'])} | "
                    f"Highest Month={item['highest_month']} | "
                    f"Highest Value={md_format_number(item['highest_value'])} | "
                    f"Lowest Month={item['lowest_month']} | "
                    f"Lowest Value={md_format_number(item['lowest_value'])} | "
                    f"Months Above Budget={', '.join(item['months_above_budget']) if item['months_above_budget'] else 'None'} | "
                    f"Months Below Budget={', '.join(item['months_below_budget']) if item['months_below_budget'] else 'None'}"
                )

    return "\n".join(
        lines
    )


# ============================================================
# 19. BUILD NATURAL-LANGUAGE-ONLY PROMPT
# ============================================================

def build_natural_language_prompt(
    question,
    analysis_type,
    context
):

    return f"""
You are the Natural Language Generation component of
a hotel financial AI agent.

The financial data has already been retrieved and
verified by Python.

Your task is ONLY to explain the supplied financial
information in natural language.

IMPORTANT OUTPUT RULES:

1. Do NOT create a table.
2. Do NOT reproduce the DataFrame.
3. Do NOT use Markdown tables.
4. Do NOT use #.
5. Do NOT use ##.
6. Do NOT use **.
7. Do NOT use Markdown headings.
8. Do NOT use bullet points.
9. Do NOT invent financial figures.
10. Do NOT modify supplied financial figures.
11. Do NOT mix records from unrelated sections.
12. Do NOT claim a cause unless the supplied data
    supports the relationship.
13. Use complete natural-language paragraphs.
14. You may use normal sentences and paragraphs only.

ANALYSIS TYPE:
{analysis_type}

USER QUESTION:
{question}

VERIFIED FINANCIAL CONTEXT:
{context}

ANALYSIS INSTRUCTIONS:

If the analysis is WITHIN_SECTION:

Explain the relevant KPIs within the section and
describe the financial performance supported by
those KPIs.

If the analysis is CROSS_SECTION:

Explain the relationship between the relevant
financial sections.

For example, when discussing revenue and profit,
describe how the supplied revenue, expenses,
departmental profit, GOP, EBITDA or total profit
figures relate to each other.

Do not claim causation unless the numerical
relationship supports the statement.

If the analysis is MONTHLY:

Describe the movement across the months.

Identify important high and low months, overall
performance, budget variances and prior forecast
variances where relevant.

If the analysis is CROSS_SECTION_MONTH:

Analyze both dimensions.

Explain how the relevant sections changed across
the months and how the sections relate to one
another.

When discussing a variance:

State the Actual value, comparison value,
absolute variance, percentage variance and
direction where these values are available.

Be concise but analytically useful.

Return ONLY the natural-language answer.
"""


# ============================================================
# 20. CLEAN LLM OUTPUT
# ============================================================

def clean_final_natural_language(
    text
):

    if not text:
        return ""

    text = str(text).strip()

    # --------------------------------------------------------
    # Remove headings
    # --------------------------------------------------------

    text = re.sub(
        r"^\s*#{1,6}\s*",
        "",
        text,
        flags=re.MULTILINE
    )

    # --------------------------------------------------------
    # Remove bold / italic Markdown
    # --------------------------------------------------------

    text = text.replace(
        "**",
        ""
    )

    text = text.replace(
        "__",
        ""
    )

    # --------------------------------------------------------
    # Remove bullet symbols
    # --------------------------------------------------------

    text = re.sub(
        r"^\s*[-*+]\s+",
        "",
        text,
        flags=re.MULTILINE
    )

    # --------------------------------------------------------
    # Remove Markdown table rows
    # --------------------------------------------------------

    cleaned_lines = []

    for line in text.splitlines():

        stripped = line.strip()

        if not stripped:
            continue

        if (
            stripped.startswith("|")
            and stripped.endswith("|")
        ):
            continue

        if re.match(
            r"^\|?[\s:-]+\|[\s|:-]*$",
            stripped
        ):
            continue

        cleaned_lines.append(
            stripped
        )

    text = "\n\n".join(
        cleaned_lines
    )

    # --------------------------------------------------------
    # Remove accidental repeated spaces
    # --------------------------------------------------------

    text = re.sub(
        r"[ \t]+",
        " ",
        text
    )

    text = re.sub(
        r"\n{3,}",
        "\n\n",
        text
    )

    return text.strip()


# ============================================================
# 21. DISPLAY FINAL RESULT
# ============================================================

def display_final_agent_result(
    result
):

    if not result:
        return

    narrative = result.get(
        "narrative",
        ""
    )

    dataframe = result.get(
        "dataframe"
    )

    # --------------------------------------------------------
    # Natural language
    # --------------------------------------------------------

    if narrative:

        text = clean_final_natural_language(
            narrative
        )

        # Escape HTML
        text = (
            text
            .replace(
                "&",
                "&amp;"
            )
            .replace(
                "<",
                "&lt;"
            )
            .replace(
                ">",
                "&gt;"
            )
        )

        # ----------------------------------------------------
        # Bold important financial terms
        # ----------------------------------------------------

        bold_terms = [

            "Actual",

            "Budget",

            "Prior Forecast",

            "Total Revenue",

            "Rooms Division",

            "F&B Division",

            "Gross Operating Profit",

            "EBITDA",

            "Total Profit",

            "variance",

            "above budget",

            "below budget",

            "above the budget",

            "below the budget"
        ]

        for term in sorted(
            bold_terms,
            key=len,
            reverse=True
        ):

            text = re.sub(

                rf"(?<![A-Za-z])"
                rf"({re.escape(term)})"
                rf"(?![A-Za-z])",

                r"<strong>\1</strong>",

                text,

                flags=re.IGNORECASE
            )

        text = text.replace(
            "\n\n",
            "<br><br>"
        )

        display(
            HTML(
                f"""
                <div style="
                    font-size: 15px;
                    line-height: 1.7;
                    margin: 10px 0 18px 0;
                ">
                    {text}
                </div>
                """
            )
        )

    # --------------------------------------------------------
    # DataFrame
    # --------------------------------------------------------

    if (
        dataframe is not None
        and not dataframe.empty
    ):

        display(
            dataframe
        )

In [ ]:
# ============================================================
# FINAL AGENT 1 CONNECTION
# ============================================================
#
# Natural-language output + pandas DataFrame
#
# The LLM does NOT generate tables.
# Python generates and displays the DataFrame.
# ============================================================


# ============================================================
# 1. FINAL RESPONSE GENERATOR
# ============================================================

def generate_agent_response(
    question,
    route,
    retrieval_mode,
    records,
    analysis=None,
    analysis_type=None
):
    """
    Final Natural Language Generation component.

    Python:
        - retrieves records
        - calculates variances
        - builds DataFrame

    LLM:
        - explains the verified financial information
        - produces natural language only
    """

    # ========================================================
    # VALIDATION
    # ========================================================

    if not records:

        return {

            "status":
                "not_found",

            "route":
                route,

            "retrieval_mode":
                retrieval_mode,

            "analysis_type":
                analysis_type,

            "narrative":
                "No relevant financial records were found.",

            "dataframe":
                pd.DataFrame(),

            "records":
                []
        }

    # ========================================================
    # DETERMINE ANALYSIS TYPE
    # ========================================================

    if analysis_type is None:

        analysis_type = (
            md_determine_analysis_type(
                question
            )
        )

    # ========================================================
    # BUILD DATAFRAME
    # ========================================================

    dataframe = (
        build_multidimensional_dataframe(
            records
        )
    )

    # ========================================================
    # BUILD VERIFIED CONTEXT
    # ========================================================

    context = (
        build_multidimensional_context(
            records,
            analysis_type,
            analysis
        )
    )

    # ========================================================
    # NATURAL LANGUAGE PROMPT
    # ========================================================

    prompt = (
        build_natural_language_prompt(
            question=question,
            analysis_type=analysis_type,
            context=context
        )
    )

    # ========================================================
    # CALL LLM
    # ========================================================

    try:

        response = client.chat.completions.create(

            model="openai/gpt-oss-20b",

            messages=[

                {
                    "role":
                        "system",

                    "content":
                        (
                            "You are the natural-language "
                            "generation component of a hotel "
                            "financial AI agent. "
                            "Use only verified financial "
                            "information supplied by Python. "
                            "Return natural language only. "
                            "Never create tables, headings, "
                            "Markdown, bullets, or bold "
                            "markers."
                        )
                },

                {
                    "role":
                        "user",

                    "content":
                        prompt
                }
            ],

            temperature=0
        )

        narrative = (

            response
            .choices[0]
            .message
            .content
            .strip()
        )

        narrative = (
            clean_final_natural_language(
                narrative
            )
        )

        return {

            "status":
                "success",

            "route":
                route,

            "retrieval_mode":
                retrieval_mode,

            "analysis_type":
                analysis_type,

            "narrative":
                narrative,

            "dataframe":
                dataframe,

            "records":
                records,

            "analysis":
                analysis
        }

    except Exception as e:

        print(
            "LLM generation error:",
            e
        )

        return {

            "status":
                "error",

            "route":
                route,

            "retrieval_mode":
                retrieval_mode,

            "analysis_type":
                analysis_type,

            "narrative":
                (
                    "The financial information was "
                    "retrieved successfully, but the "
                    "natural-language analysis could "
                    "not be generated."
                ),

            "dataframe":
                dataframe,

            "records":
                records,

            "analysis":
                analysis
        }


# ============================================================
# 2. MASTER ask_agent()
# ============================================================

def ask_agent(
    question,
    debug=True
):
    """
    Master Agent 1.

    Supports:

        WITHIN_SECTION
        CROSS_SECTION
        MONTHLY
        CROSS_SECTION_MONTH

    and retains:

        RETRIEVAL
        VARIANCE
        COMPARISON
        DIAGNOSTIC
    """

    question = str(
        question
    ).strip()

    # ========================================================
    # EMPTY QUESTION
    # ========================================================

    if not question:

        return {

            "status":
                "error",

            "route":
                None,

            "retrieval_mode":
                None,

            "analysis_type":
                None,

            "narrative":
                "Please enter a financial question.",

            "dataframe":
                pd.DataFrame()
        }

    # ========================================================
    # ANALYSIS TYPE
    # ========================================================

    analysis_type = (
        md_determine_analysis_type(
            question
        )
    )

    # ========================================================
    # MONTH
    # ========================================================

    requested_month = (
        md_detect_month(
            question
        )
    )

    # ========================================================
    # ROUTE
    # ========================================================

    route = route_financial_query(
        question
    )

    # ========================================================
    # DEBUG
    # ========================================================

    if debug:

        print("\n" + "=" * 80)

        print(
            "FINANCIAL AI AGENT"
        )

        print("=" * 80)

        print(
            "QUESTION:",
            question
        )

        print(
            "ROUTE:",
            route
        )

        print(
            "ANALYSIS TYPE:",
            analysis_type
        )

        print(
            "REQUESTED MONTH:",
            requested_month
        )

        print("=" * 80)

    # ========================================================
    # MULTI-DIMENSIONAL ANALYSIS
    # ========================================================
    #
    # This is checked FIRST.
    #
    # This means questions requiring:
    #
    #   section analysis
    #   cross-section analysis
    #   monthly analysis
    #   cross-section/month analysis
    #
    # are handled by the new analytical layer.
    # ========================================================

    multidimensional_keywords = [

        "across",
        "throughout",
        "over the year",
        "monthly",
        "months",
        "each month",
        "every month",
        "trend",
        "performance",
        "contribution",
        "contribute",
        "affect",
        "affected",
        "impact",
        "relationship",
        "factors",
        "drivers",
        "why did",
        "why was",
        "overall",
        "analyze",
        "analysis"
    ]

    q_normalized = md_normalize(
        question
    )

    requires_multidimensional = (

        analysis_type != "WITHIN_SECTION"

        or any(
            term in q_normalized
            for term in multidimensional_keywords
        )
    )

    # ========================================================
    # MULTI-DIMENSIONAL PATH
    # ========================================================

    if requires_multidimensional:

        multidimensional_records = (
            retrieve_multidimensional_records(
                question,
                financial_records,
                debug=debug
            )
        )

        if multidimensional_records:

            if analysis_type in [
                "MONTHLY",
                "CROSS_SECTION_MONTH"
            ]:

                analysis = (
                    analyze_multidimensional_monthly(
                        multidimensional_records
                    )
                )

            else:

                analysis = None

            return generate_agent_response(

                question=question,

                route=route,

                retrieval_mode=(
                    analysis_type
                ),

                records=(
                    multidimensional_records
                ),

                analysis=analysis,

                analysis_type=(
                    analysis_type
                )
            )

    # ========================================================
    # ORIGINAL SINGLE RECORD RETRIEVAL
    # ========================================================

    if route == "RETRIEVAL":

        retrieved = retrieve_best_record(

            question,

            financial_records,

            target_period=(
                requested_month
            ),

            default_full_year=(
                requested_month is None
            ),

            debug=debug
        )

        if not retrieved:

            return {

                "status":
                    "not_found",

                "route":
                    route,

                "retrieval_mode":
                    "SINGLE",

                "analysis_type":
                    "WITHIN_SECTION",

                "narrative":
                    (
                        "No matching financial "
                        "record was found."
                    ),

                "dataframe":
                    pd.DataFrame()
            }

        return generate_agent_response(

            question=question,

            route=route,

            retrieval_mode="SINGLE",

            records=retrieved,

            analysis=None,

            analysis_type="WITHIN_SECTION"
        )

    # ========================================================
    # VARIANCE
    # ========================================================

    if route == "VARIANCE":

        retrieved = retrieve_best_record(

            question,

            financial_records,

            target_period=(
                requested_month
            ),

            default_full_year=(
                requested_month is None
            ),

            debug=debug
        )

        if not retrieved:

            return {

                "status":
                    "not_found",

                "route":
                    route,

                "retrieval_mode":
                    "SINGLE",

                "analysis_type":
                    "WITHIN_SECTION",

                "narrative":
                    (
                        "No matching financial "
                        "record was found."
                    ),

                "dataframe":
                    pd.DataFrame()
            }

        return generate_agent_response(

            question=question,

            route=route,

            retrieval_mode="SINGLE",

            records=retrieved,

            analysis=None,

            analysis_type="WITHIN_SECTION"
        )

    # ========================================================
    # COMPARISON
    # ========================================================

    if route == "COMPARISON":

        comparison_records = (
            retrieve_comparison_records(
                question,
                financial_records,
                debug=debug
            )
        )

        if comparison_records:

            comparison_analysis = (
                analyze_comparison(
                    comparison_records,
                    question
                )
            )

            return generate_agent_response(

                question=question,

                route=route,

                retrieval_mode=(
                    "CROSS_SECTION"
                ),

                records=(
                    comparison_records
                ),

                analysis=(
                    comparison_analysis
                ),

                analysis_type=(
                    "CROSS_SECTION"
                )
            )

        return {

            "status":
                "not_found",

            "route":
                route,

            "retrieval_mode":
                "COMPARISON_SET",

            "analysis_type":
                "CROSS_SECTION",

            "narrative":
                (
                    "No comparable financial "
                    "records were found."
                ),

            "dataframe":
                pd.DataFrame()
        }

    # ========================================================
    # DIAGNOSTIC
    # ========================================================

    if route == "DIAGNOSTIC":

        diagnostic_result = diagnostic_agent(

            question,

            max_iterations=2,

            debug=debug
        )

        diagnostic_records = (
            diagnostic_result.get(
                "records",
                []
            )
        )

        if diagnostic_records:

            diagnostic_mode = (
                diagnostic_result.get(
                    "mode",
                    "DIAGNOSTIC"
                )
            )

            diagnostic_analysis = (
                diagnostic_result.get(
                    "analysis"
                )
            )

            return generate_agent_response(

                question=question,

                route=route,

                retrieval_mode=(
                    diagnostic_mode
                ),

                records=(
                    diagnostic_records
                ),

                analysis=(
                    diagnostic_analysis
                ),

                analysis_type=(
                    analysis_type
                )
            )

        return {

            "status":
                "not_found",

            "route":
                route,

            "retrieval_mode":
                "DIAGNOSTIC",

            "analysis_type":
                analysis_type,

            "narrative":
                (
                    "Insufficient financial "
                    "context was found."
                ),

            "dataframe":
                pd.DataFrame()
        }

    # ========================================================
    # FALLBACK
    # ========================================================

    return {

        "status":
            "error",

        "route":
            route,

        "retrieval_mode":
            None,

        "analysis_type":
            analysis_type,

        "narrative":
            (
                "The agent could not determine "
                "a valid analysis route."
            ),

        "dataframe":
            pd.DataFrame()
    }

In [ ]:
# ============================================================
# TEST 1 — DIRECT RETRIEVAL
# ============================================================

result = ask_agent(
    "What was the total revenue?",
    debug=True
)

display_final_agent_result(
    result
)

In [ ]:
# ============================================================
# TEST 2 — WITHIN SECTION
# ============================================================

result = ask_agent(
    "How did the Rooms Statistics perform?",
    debug=True
)

display_final_agent_result(
    result
)

In [ ]:
# ============================================================
# TEST 3 — CROSS SECTION
# ============================================================

result = ask_agent(
    "How did Rooms Division and F&B Division contribute to total revenue?",
    debug=True
)

display_final_agent_result(
    result
)

In [ ]:
# ============================================================
# TEST 4 — MONTHLY ANALYSIS
# ============================================================

result = ask_agent(
    "How did total revenue perform across the 12 months?",
    debug=True
)

display_final_agent_result(
    result
)

In [ ]:
# ============================================================
# TEST 5 — CROSS SECTION + CROSS MONTH
# ============================================================

result = ask_agent(
    "How did Rooms Division performance affect total revenue throughout the year?",
    debug=True
)

display_final_agent_result(
    result
)

In [ ]:
# ============================================================
# TEST 6 — F&B MONTHLY ANALYSIS
# ============================================================

result = ask_agent(
    "How did F&B performance change across the year?",
    debug=True
)

display_final_agent_result(
    result
)

In [ ]:
# ============================================================
# TEST 7 — PROFITABILITY CHAIN
# ============================================================

result = ask_agent(
    "How did revenue, departmental expenses and undistributed expenses affect GOP?",
    debug=True
)

display_final_agent_result(
    result
)

In [ ]:
# ============================================================
# TEST 8 — EBITDA ANALYSIS
# ============================================================

result = ask_agent(
    "What factors contributed to the change in EBITDA?",
    debug=True
)

display_final_agent_result(
    result
)

In [ ]:
# ============================================================
# TEST 9 — MONTHLY PROFITABILITY
# ============================================================

result = ask_agent(
    "How did EBITDA change from January to December?",
    debug=True
)

display_final_agent_result(
    result
)

In [ ]:
# ============================================================
# TEST 10 — BUDGET COMPARISON
# ============================================================

result = ask_agent(
    "How did total revenue compare with the budget?",
    debug=True
)

display_final_agent_result(
    result
)

In [ ]:
# ============================================================
# TEST 11 — MONTHLY BUDGET PERFORMANCE
# ============================================================

result = ask_agent(
    "Which months were below budget for total revenue?",
    debug=True
)

display_final_agent_result(
    result
)

In [ ]:
# ============================================================
# TEST 12 — ROOMS RELATIONSHIP
# ============================================================

result = ask_agent(
    "What was the relationship between occupancy, average rate and RevPAR across the year?",
    debug=True
)

display_final_agent_result(
    result
)

In [ ]:
# ============================================================
# TEST 13 — OVERALL FINANCIAL ANALYSIS
# ============================================================

result = ask_agent(
    "Analyze the hotel's overall financial performance across the year.",
    debug=True
)

display_final_agent_result(
    result
)

In [ ]:
# ============================================================
# TEST 14 — Performance of RevPAR
# ============================================================

result = ask_agent(
    "Why is RevPAR underperforming?",
    debug=True
)

display_final_agent_result(
    result
)